# MongoDB

## 1. Database relazionali e NoSQL

Un database relazionale (MySQL, PostgreSQL, ...) organizza i dati in tabelle. Prima di inserire qualsiasi dato bisogna definire lo schema, cioè quali colonne ha ogni tabella e di che tipo sono. I dati collegati tra loro stanno in tabelle diverse e si rimettono insieme con le JOIN.

I database NoSQL nascono per gestire dati meno rigidi. MongoDB è un database NoSQL *a documenti*: invece delle righe di una tabella salva documenti simili a oggetti JSON, che possono contenere liste e altri oggetti al loro interno.

Per esempio, uno studente con i suoi voti in un database relazionale richiede due tabelle (`studenti` e `voti`) collegate da un id. In MongoDB è un unico documento:

```json
{
  "nome": "Mario",
  "eta": 19,
  "voti": [
    {"materia": "Python", "voto": 8},
    {"materia": "Reti", "voto": 7}
  ]
}
```

Questo è il motivo per cui MongoDB si usa spesso insieme alle API REST: i dati che un client manda in JSON si possono salvare quasi così come sono, e allo stesso modo si possono rispedire indietro.

MongoDB non è sempre la scelta migliore: quando i dati hanno moltissime relazioni tra loro (pensate alla contabilità di un'azienda) un database relazionale resta più adatto.

## 2. Documenti, collezioni, database

Un **documento** è un insieme di coppie campo-valore. I valori possono essere stringhe, numeri, booleani, liste o altri documenti. Ogni documento ha un campo `_id` che lo identifica in modo univoco: se non lo indichiamo noi, MongoDB lo crea automaticamente.

I documenti sono raggruppati in **collezioni**, che corrispondono alle tabelle. A differenza di una tabella però, una collezione non impone uno schema: documenti della stessa collezione possono avere campi diversi.

Le collezioni a loro volta stanno in un **database**, e un server MongoDB può ospitare più database:

```
server MongoDB
 └── database "scuola"
      ├── collezione "studenti"
      │    ├── documento
      │    └── documento
      └── collezione "corsi"
           └── documento
```

| SQL | MongoDB |
|---|---|
| tabella | collezione |
| riga | documento |
| colonna | campo |

## 3. Collegarsi con PyMongo

Per avere un server MongoDB in locale basta avviare un container:

```bash
docker run -d --name mongo -p 27017:27017 mongo
```

Da Python usiamo la libreria **PyMongo** (`pip install pymongo`) e in particolare il suo client asincrono, `AsyncMongoClient`.

In [2]:
from pymongo import AsyncMongoClient

client = AsyncMongoClient("mongodb://localhost:27017")   # collegamento al server
db = client["scuola"]                                    # database
studenti = db["studenti"]                                # collezione

await client.admin.command("ping")   # se il server risponde: {'ok': 1.0}

{'ok': 1.0}

Queste righe non creano nulla sul server: il database e la collezione vengono creati automaticamente quando inseriamo il primo documento.

### Dati di esempio

La cella seguente svuota la collezione e inserisce sei studenti. Possiamo rieseguirla ogni volta che vogliamo ripartire dai dati iniziali.

Notate che i documenti non sono tutti uguali: Luca e Paolo non hanno l'email, Anna non ha il campo `voti`, Marco ha una lista di voti vuota.

In [3]:
await studenti.drop()   # cancella la collezione, se esiste

await studenti.insert_many([
    {"nome": "Giulia", "eta": 18, "citta": "Parma", "email": "giulia@example.it",
     "corsi": ["Python", "Web"],
     "voti": [{"materia": "Python", "voto": 9}, {"materia": "Web", "voto": 7}]},
    {"nome": "Luca", "eta": 19, "citta": "Milano",
     "corsi": ["Python", "Reti"],
     "voti": [{"materia": "Python", "voto": 5}, {"materia": "Reti", "voto": 9}]},
    {"nome": "Sara", "eta": 18, "citta": "Parma", "email": "sara@example.it",
     "corsi": ["Python", "Sistemi", "Database"],
     "voti": [{"materia": "Python", "voto": 8}]},
    {"nome": "Marco", "eta": 20, "citta": "Bologna", "email": "marco@example.it",
     "corsi": ["Reti", "Database"],
     "voti": []},
    {"nome": "Anna", "eta": 17, "citta": "Roma", "email": "anna@example.it",
     "corsi": ["Web"]},
    {"nome": "Paolo", "eta": 19, "citta": "Milano",
     "corsi": ["Python", "Reti", "Database"],
     "voti": [{"materia": "Python", "voto": 6}, {"materia": "Database", "voto": 10}]},
])

InsertManyResult([ObjectId('6afb5e2b485ee1cec9ca7801'), ObjectId('6afb5e2b485ee1cec9ca7802'), ObjectId('6afb5e2b485ee1cec9ca7803'), ObjectId('6afb5e2b485ee1cec9ca7804'), ObjectId('6afb5e2b485ee1cec9ca7805'), ObjectId('6afb5e2b485ee1cec9ca7806')], acknowledged=True)

## 4. Leggere i documenti

### 4.1 `find_one`

`find_one(filtro)` restituisce il primo documento che corrisponde al filtro, oppure `None` se non ne trova nessuno.

Il filtro è un dizionario: `{"citta": "Milano"}` significa "i documenti in cui il campo `citta` vale Milano". Il filtro vuoto `{}` corrisponde a tutti i documenti.

In Python il documento restituito è un normale dizionario.

In [26]:
studente = await studenti.find_one({"citta": "Milano","eta": 19})
print(studente)
print(type(studente))

print(await studenti.find_one({"citta": "Napoli"}))   # nessun risultato: None

{'_id': ObjectId('6afb5e2b485ee1cec9ca7806'), 'nome': 'Paolo', 'eta': 19, 'citta': 'Milano', 'corsi': ['Python', 'Reti', 'Database'], 'voti': [{'materia': 'Python', 'voto': 6}, {'materia': 'Database', 'voto': 10}]}
<class 'dict'>
None


### 4.2 `find` e i cursori

Per ottenere tutti i documenti che corrispondono a un filtro si usa `find(filtro)`. A differenza di `find_one`, però, `find` non restituisce direttamente i documenti ma un **cursore**.

Un cursore è un oggetto che rappresenta il risultato della query e permette di leggerlo un documento alla volta, un po' come si legge un file riga per riga. Quando creiamo il cursore la query non è ancora stata inviata: parte solo quando iniziamo a leggerlo. Per questo `find()` si scrive senza `await`.

Il server non manda tutti i risultati in un colpo solo, ma a blocchi: il cursore ne riceve un gruppo e, quando lo abbiamo letto, chiede il successivo. In questo modo si possono scorrere anche moltissimi documenti senza caricarli tutti in memoria. Dato che ogni lettura può richiedere di aspettare il server, il cursore si scorre con `async for`.

In [5]:
cursore = studenti.find({"citta": "Parma"})   # la query non è ancora partita
print(cursore)

async for studente in cursore:                # adesso sì
    print(studente["nome"])

Giulia
Sara


Un cursore si può leggere una sola volta. Una volta arrivato alla fine è esaurito, e se proviamo a scorrerlo di nuovo non otteniamo niente. Per rileggere i dati bisogna eseguire di nuovo `find()`.

In [6]:
async for studente in cursore:   # stesso cursore di prima: non stampa nulla
    print(studente["nome"])

print("fine")

fine


Se vogliamo tutti i risultati in una lista possiamo usare `to_list()`, che legge l'intero cursore. È comodo, ma carica tutto in memoria: va bene quando i risultati sono pochi.

In [7]:
lista = await studenti.find({"citta": "Parma"}).to_list()
print(len(lista), "studenti")
print(lista)

2 studenti
[{'_id': ObjectId('6afb5e2b485ee1cec9ca7801'), 'nome': 'Giulia', 'eta': 18, 'citta': 'Parma', 'email': 'giulia@example.it', 'corsi': ['Python', 'Web'], 'voti': [{'materia': 'Python', 'voto': 9}, {'materia': 'Web', 'voto': 7}]}, {'_id': ObjectId('6afb5e2b485ee1cec9ca7803'), 'nome': 'Sara', 'eta': 18, 'citta': 'Parma', 'email': 'sara@example.it', 'corsi': ['Python', 'Sistemi', 'Database'], 'voti': [{'materia': 'Python', 'voto': 8}]}]


Per gli esempi seguenti usiamo una piccola funzione che stampa tutti i documenti di un cursore:

In [8]:
async def stampa(cursore):
    async for documento in cursore:
        print(documento)

### 4.3 Scegliere i campi (proiezione)

`find` e `find_one` accettano un secondo argomento, la **proiezione**, che indica quali campi restituire: `1` per includere un campo, `0` per escluderlo. Il campo `_id` viene sempre restituito, a meno di escluderlo esplicitamente con `"_id": 0`.

In [9]:
await stampa(studenti.find({"citta": "Milano"}, {"nome": 1, "eta": 1}))
print()
await stampa(studenti.find({"citta": "Milano"}, {"_id": 0, "nome": 1, "eta": 1}))

{'_id': ObjectId('6afb5e2b485ee1cec9ca7802'), 'nome': 'Luca', 'eta': 19}
{'_id': ObjectId('6afb5e2b485ee1cec9ca7806'), 'nome': 'Paolo', 'eta': 19}

{'nome': 'Luca', 'eta': 19}
{'nome': 'Paolo', 'eta': 19}


### 4.4 Ordinare e limitare i risultati

Il cursore ha alcuni metodi che modificano il risultato della query. Si scrivono dopo `find()`, prima di leggere il cursore, e si possono concatenare:

- `sort("campo", 1)` ordina in modo crescente (`-1` per decrescente);
- `limit(n)` restituisce al massimo `n` documenti;
- `skip(n)` salta i primi `n` documenti.

`skip` e `limit` insieme servono per la paginazione: pagina 1, pagina 2, e così via.

In [10]:
# i tre studenti più grandi
await stampa(studenti.find({}, {"_id": 0, "nome": 1, "eta": 1}).sort("eta", -1).limit(3))
print()

# ordine alfabetico, seconda "pagina" da tre studenti
await stampa(studenti.find({}, {"_id": 0, "nome": 1}).sort("nome", 1).skip(3).limit(3))

{'nome': 'Marco', 'eta': 20}
{'nome': 'Paolo', 'eta': 19}
{'nome': 'Luca', 'eta': 19}

{'nome': 'Marco'}
{'nome': 'Paolo'}
{'nome': 'Sara'}


### 4.5 Contare

`count_documents(filtro)` restituisce quanti documenti corrispondono al filtro. È un metodo della collezione, non del cursore.

In [11]:
print(await studenti.count_documents({}))
print(await studenti.count_documents({"citta": "Parma"}))

6
2


### 4.6 Cercare per `_id`

Abbiamo visto che ogni documento ha un campo `_id`. Quando è MongoDB a generarlo, il suo valore è un `ObjectId`, un tipo speciale che **non è una stringa**.

Questo è importante con le API REST: un id che arriva in un URL (per esempio `/studenti/6905da48c0be...`) è una stringa, e cercandolo così com'è non si trova niente. Bisogna prima convertirlo con `ObjectId()`. Al contrario, per inviare un id in una risposta JSON lo si converte in stringa con `str()`.

In [12]:
from bson import ObjectId

giulia = await studenti.find_one({"nome": "Giulia"})
id_stringa = str(giulia["_id"])
print(id_stringa)

print(await studenti.find_one({"_id": id_stringa}))             # None
print(await studenti.find_one({"_id": ObjectId(id_stringa)}))   # trovato

6afb5e2b485ee1cec9ca7801
None
{'_id': ObjectId('6afb5e2b485ee1cec9ca7801'), 'nome': 'Giulia', 'eta': 18, 'citta': 'Parma', 'email': 'giulia@example.it', 'corsi': ['Python', 'Web'], 'voti': [{'materia': 'Python', 'voto': 9}, {'materia': 'Web', 'voto': 7}]}


## 5. Operatori di query

Finora i filtri controllavano solo l'uguaglianza. Per esprimere condizioni diverse si usano gli **operatori**, che iniziano con `$` e si scrivono così:

```python
{"campo": {"$operatore": valore}}
```

### 5.1 Operatori di confronto

- `$gt` maggiore, `$gte` maggiore o uguale
- `$lt` minore, `$lte` minore o uguale
- `$ne` diverso
- `$in` uguale a uno dei valori di una lista, `$nin` a nessuno

Su uno stesso campo si possono mettere più operatori, per esempio per indicare un intervallo.

In [13]:
print("più di 18 anni:")
await stampa(studenti.find({"eta": {"$gt": 18}}, {"_id": 0, "nome": 1, "eta": 1}))

print("\ntra 18 e 19 anni:")
await stampa(studenti.find({"eta": {"$gte": 18, "$lte": 19}}, {"_id": 0, "nome": 1, "eta": 1}))

print("\ndi Parma o di Bologna:")
await stampa(studenti.find({"citta": {"$in": ["Parma", "Bologna"]}}, {"_id": 0, "nome": 1, "citta": 1}))

più di 18 anni:
{'nome': 'Luca', 'eta': 19}
{'nome': 'Marco', 'eta': 20}
{'nome': 'Paolo', 'eta': 19}

tra 18 e 19 anni:
{'nome': 'Giulia', 'eta': 18}
{'nome': 'Luca', 'eta': 19}
{'nome': 'Sara', 'eta': 18}
{'nome': 'Paolo', 'eta': 19}

di Parma o di Bologna:
{'nome': 'Giulia', 'citta': 'Parma'}
{'nome': 'Sara', 'citta': 'Parma'}
{'nome': 'Marco', 'citta': 'Bologna'}


### 5.2 Operatori logici

Se un filtro contiene più campi, devono essere veri tutti: è un AND implicito.

Per un OR si usa `$or`, che riceve una lista di condizioni: basta che una sia vera.

In [14]:
print("18 anni e di Parma:")
await stampa(studenti.find({"eta": 18, "citta": "Parma"}, {"_id": 0, "nome": 1}))

print("\nminorenni oppure con più di 19 anni:")
await stampa(studenti.find({"$or": [{"eta": {"$lt": 18}}, {"eta": {"$gt": 19}}]}, {"_id": 0, "nome": 1, "eta": 1}))

18 anni e di Parma:
{'nome': 'Giulia'}
{'nome': 'Sara'}

minorenni oppure con più di 19 anni:
{'nome': 'Marco', 'eta': 20}
{'nome': 'Anna', 'eta': 17}


### 5.3 `$exists` e `$regex`

Visto che i documenti possono avere campi diversi, è utile poter cercare quelli che hanno (o non hanno) un certo campo: lo fa `$exists`.

`$regex` cerca le stringhe che corrispondono a un'espressione regolare. L'opzione `"i"` ignora la differenza tra maiuscole e minuscole.

In [15]:
print("senza email:")
await stampa(studenti.find({"email": {"$exists": False}}, {"_id": 0, "nome": 1}))

print("\nnome che inizia con 'ma':")
await stampa(studenti.find({"nome": {"$regex": "^ma", "$options": "i"}}, {"_id": 0, "nome": 1}))

senza email:
{'nome': 'Luca'}
{'nome': 'Paolo'}

nome che inizia con 'ma':
{'nome': 'Marco'}


### 5.4 Liste e documenti annidati

Quando un campo è una lista, il filtro `{"corsi": "Python"}` significa "la lista contiene Python". Con `$all` si chiede che la lista contenga tutti i valori indicati.

Per cercare dentro un documento annidato si usa la notazione con il punto: `"voti.voto"` indica il campo `voto` dei documenti che stanno in `voti`.

In [16]:
print("iscritti a Python:")
await stampa(studenti.find({"corsi": "Python"}, {"_id": 0, "nome": 1}))

print("\niscritti sia a Python sia a Reti:")
await stampa(studenti.find({"corsi": {"$all": ["Python", "Reti"]}}, {"_id": 0, "nome": 1}))

print("\ncon almeno un 10:")
await stampa(studenti.find({"voti.voto": 10}, {"_id": 0, "nome": 1}))

iscritti a Python:
{'nome': 'Giulia'}
{'nome': 'Luca'}
{'nome': 'Sara'}
{'nome': 'Paolo'}

iscritti sia a Python sia a Reti:
{'nome': 'Luca'}
{'nome': 'Paolo'}

con almeno un 10:
{'nome': 'Paolo'}


### 5.5 `$elemMatch`

Proviamo a cercare chi ha almeno 8 in Python. Il primo tentativo che viene in mente è questo:

In [17]:
await stampa(studenti.find({"voti.materia": "Python", "voti.voto": {"$gte": 8}}, {"_id": 0, "nome": 1, "voti": 1}))

{'nome': 'Giulia', 'voti': [{'materia': 'Python', 'voto': 9}, {'materia': 'Web', 'voto': 7}]}
{'nome': 'Luca', 'voti': [{'materia': 'Python', 'voto': 5}, {'materia': 'Reti', 'voto': 9}]}
{'nome': 'Sara', 'voti': [{'materia': 'Python', 'voto': 8}]}
{'nome': 'Paolo', 'voti': [{'materia': 'Python', 'voto': 6}, {'materia': 'Database', 'voto': 10}]}


Il risultato contiene anche Luca e Paolo, che in Python hanno 5 e 6. Il motivo è che le due condizioni vengono verificate separatamente sulla lista: Luca ha un voto in Python e ha *un altro* voto maggiore di 8 (il 9 in Reti), quindi le soddisfa entrambe.

Per chiedere che le condizioni valgano per **lo stesso elemento** della lista si usa `$elemMatch`:

In [18]:
await stampa(studenti.find({"voti": {"$elemMatch": {"materia": "Python", "voto": {"$gte": 8}}}}, {"_id": 0, "nome": 1, "voti": 1}))

{'nome': 'Giulia', 'voti': [{'materia': 'Python', 'voto': 9}, {'materia': 'Web', 'voto': 7}]}
{'nome': 'Sara', 'voti': [{'materia': 'Python', 'voto': 8}]}


## 6. Inserire documenti

`insert_one(documento)` inserisce un documento e restituisce un oggetto il cui attributo `inserted_id` contiene l'`_id` assegnato.

`insert_many(lista)` inserisce più documenti; gli id si trovano in `inserted_ids`.

In [19]:
risultato = await studenti.insert_one({"nome": "Chiara", "eta": 18, "corsi": ["Python"]})
print(risultato.inserted_id)

risultato = await studenti.insert_many([
    {"nome": "Davide", "eta": 19},
    {"nome": "Elena", "eta": 18, "citta": "Parma"},
])
print(risultato.inserted_ids)

6afb5e2c485ee1cec9ca7807
[ObjectId('6afb5e2c485ee1cec9ca7808'), ObjectId('6afb5e2c485ee1cec9ca7809')]


## 7. Modificare documenti

`update_one(filtro, modifica)` modifica il primo documento che corrisponde al filtro, `update_many(filtro, modifica)` li modifica tutti.

La modifica si scrive con gli operatori di aggiornamento:

- `$set` assegna un valore a un campo (se il campo non c'è, lo crea)
- `$unset` elimina un campo
- `$inc` incrementa un numero (con un valore negativo lo decrementa)
- `$push` aggiunge un elemento a una lista
- `$addToSet` aggiunge un elemento solo se non è già presente
- `$pull` toglie un elemento da una lista

Il risultato indica quanti documenti corrispondevano al filtro (`matched_count`) e quanti sono stati effettivamente modificati (`modified_count`). I due numeri possono essere diversi: se il valore era già quello, il documento viene trovato ma non modificato.

In [20]:
risultato = await studenti.update_one({"nome": "Luca"}, {"$set": {"email": "luca@example.it"}})
print(risultato.matched_count, risultato.modified_count)   # 1 1

risultato = await studenti.update_one({"nome": "Luca"}, {"$set": {"email": "luca@example.it"}})
print(risultato.matched_count, risultato.modified_count)   # 1 0: l'email era già quella

# più operatori nella stessa modifica
await studenti.update_one({"nome": "Luca"}, {"$inc": {"eta": 1}, "$unset": {"citta": ""}})
print(await studenti.find_one({"nome": "Luca"}, {"_id": 0}))

1 1
1 0
{'nome': 'Luca', 'eta': 20, 'corsi': ['Python', 'Reti'], 'voti': [{'materia': 'Python', 'voto': 5}, {'materia': 'Reti', 'voto': 9}], 'email': 'luca@example.it'}


In [21]:
# operazioni sulle liste
await studenti.update_one({"nome": "Anna"}, {"$push": {"corsi": "Python"}})
print(await studenti.find_one({"nome": "Anna"}, {"_id": 0, "corsi": 1}))

await studenti.update_one({"nome": "Anna"}, {"$addToSet": {"corsi": "Python"}})   # già presente: niente
print(await studenti.find_one({"nome": "Anna"}, {"_id": 0, "corsi": 1}))

await studenti.update_one({"nome": "Anna"}, {"$pull": {"corsi": "Web"}})
print(await studenti.find_one({"nome": "Anna"}, {"_id": 0, "corsi": 1}))

{'corsi': ['Web', 'Python']}
{'corsi': ['Web', 'Python']}
{'corsi': ['Python']}


In [22]:
# tutti gli studenti di Parma
risultato = await studenti.update_many({"citta": "Parma"}, {"$set": {"regione": "Emilia-Romagna"}})
print(risultato.modified_count, "documenti modificati")

3 documenti modificati


La modifica deve sempre usare un operatore. Se scriviamo `{"eta": 21}` invece di `{"$set": {"eta": 21}}`, PyMongo dà errore:

In [23]:
try:
    await studenti.update_one({"nome": "Luca"}, {"eta": 21})
except ValueError as errore:
    print("Errore:", errore)

Errore: update only works with $ operators


## 8. Cancellare documenti

`delete_one(filtro)` cancella il primo documento che corrisponde al filtro, `delete_many(filtro)` li cancella tutti. Il numero di documenti cancellati è in `deleted_count`.

Attenzione: `delete_many({})`, con il filtro vuoto, cancella tutti i documenti della collezione.

In [24]:
risultato = await studenti.delete_one({"nome": "Chiara"})
print(risultato.deleted_count)

risultato = await studenti.delete_many({"eta": {"$lt": 18}})
print(risultato.deleted_count)

print(await studenti.count_documents({}), "documenti rimasti")

1
1
7 documenti rimasti
